# B2.4 - Risk-category classifier: M4 DeBERTa-v3-large, 1 seed (first to cut) (FinSight)

Fine-tunes `microsoft/deberta-v3-large` to put each risk factor in one of 10 categories. **Runs on Kaggle (T4, fp16)**;
nothing trains on the laptop. Data: the private dataset `finsight-risk-classifier` (teacher labels,
split 90/10 **by company**, so no company is on both sides).

- Class-weighted cross-entropy, linear warm-up and decay, max_length 384.
- The best epoch is kept by **dev macro-F1**. Gold-150 is never used here.
- A seed whose `metrics-<seed>.json` is in an attached earlier output is skipped (resume).
- Outputs: `metrics-<seed>.json`, `dev-predictions-<seed>.json` and `seed-<seed>/final/` (weights and
  `tokenizer.json`, which the ONNX export needs).
- Labels come from the teacher model (AI-made, `label_source`). Report a seed count with every number.


In [ ]:
# ---- parameters ------------------------------------------------------------------------------
MODEL = 'microsoft/deberta-v3-large'
DATASET = 'finsight-risk-classifier'
SEEDS = [42]
EPOCHS = 3
LEARNING_RATE = 1e-05
BATCH_SIZE = 4
GRAD_ACCUM = 4
MAX_LEN = 384
SMOKE = False  # True: 200 train / 100 dev rows, 1 epoch, first seed only
OUT_ROOT = "/kaggle/working/classifier"


In [ ]:
"""Classifier scores (B04 E16): macro-F1, per-class F1, confusion. Standard library only.

``scripts/make_classifier_notebooks.py`` pastes this file into the Kaggle notebooks, so the
epoch the notebook keeps is chosen with exactly the metric the laptop reports.
"""

from __future__ import annotations

import hashlib
from collections.abc import Sequence
from typing import Any


def scores(gold: Sequence[str], pred: Sequence[str], labels: Sequence[str]) -> dict[str, Any]:
    """Macro-F1 over ``labels`` (a class absent from gold and pred counts as F1 0 only if
    it was predicted or expected; classes with no support and no predictions are left out)."""
    if len(gold) != len(pred):
        raise ValueError("gold and pred differ in length")
    index = {lab: i for i, lab in enumerate(labels)}
    confusion = [[0] * len(labels) for _ in labels]
    for g, p in zip(gold, pred, strict=True):
        confusion[index[g]][index[p]] += 1
    per_class: dict[str, dict[str, float | int]] = {}
    for lab, i in index.items():
        tp = confusion[i][i]
        fp = sum(confusion[j][i] for j in range(len(labels))) - tp
        fn = sum(confusion[i]) - tp
        if tp + fp + fn == 0:
            continue
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        per_class[lab] = {"precision": prec, "recall": rec, "f1": f1, "support": tp + fn}
    n = len(gold)
    return {
        "n": n,
        "accuracy": sum(confusion[i][i] for i in range(len(labels))) / n if n else 0.0,
        "macro_f1": sum(c["f1"] for c in per_class.values()) / len(per_class) if per_class else 0.0,
        "per_class": per_class,
        "labels": list(labels),
        "confusion": confusion,
    }


def company_bucket(company: str, seed: int = 2026) -> float:
    """A stable number in [0, 1) per company, so a split never puts one company on both sides."""
    digest = hashlib.sha256(f"{seed}:{company.strip().lower()}".encode()).hexdigest()
    return int(digest[:12], 16) / 16**12


In [ ]:
import json, math, random, shutil, time
from pathlib import Path

import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    DATA_DIR = next(p.parent for p in Path("/kaggle/input").rglob("labels.json"))
else:
    DATA_DIR, OUT_ROOT = Path("data/processed/kaggle") / "risk-classifier", "classifier_out"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABELS = json.loads((DATA_DIR / "labels.json").read_text(encoding="utf-8"))

def read(name):
    with (DATA_DIR / f"{name}.jsonl").open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train, dev = read("train"), read("dev")
assert not {r["company"] for r in train} & {r["company"] for r in dev}, "company on both sides"
if SMOKE:
    train, dev, EPOCHS, SEEDS = train[:200], dev[:100], 1, SEEDS[:1]
print(len(train), "train |", len(dev), "dev | device:", DEVICE, "x", torch.cuda.device_count())

tokenizer = AutoTokenizer.from_pretrained(MODEL)
counts = np.bincount([LABELS.index(r["label"]) for r in train], minlength=len(LABELS))
class_weights = torch.tensor(len(train) / (len(LABELS) * np.maximum(counts, 1)), dtype=torch.float, device=DEVICE)
use_amp = DEVICE.type == "cuda"

def encode(rows):
    enc = tokenizer([r["text"] for r in rows], truncation=True, max_length=MAX_LEN, padding=True,
                    return_tensors="pt")
    return {k: v.to(DEVICE) for k, v in enc.items()}

def dev_probs(model, rows, size=32):
    model.eval()
    out = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
        for lo in range(0, len(rows), size):
            logits = model(**encode(rows[lo:lo + size])).logits.float()
            out += torch.softmax(logits, -1).cpu().tolist()
    return out

def earlier_metrics(seed):
    if not ON_KAGGLE:
        return None
    hit = next(Path("/kaggle/input").rglob(f"metrics-{seed}.json"), None)
    return json.loads(hit.read_text(encoding="utf-8")) if hit else None

def run_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)),
        label2id={l: i for i, l in enumerate(LABELS)}).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    steps = EPOCHS * math.ceil(len(train) / (BATCH_SIZE * GRAD_ACCUM))
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
    out_dir = Path(OUT_ROOT) / f"seed-{seed}"
    best, per_epoch, t0 = (-1.0, None), [], time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        order = random.sample(range(len(train)), len(train))
        losses = []
        for i, lo in enumerate(range(0, len(order), BATCH_SIZE)):
            rows = [train[j] for j in order[lo:lo + BATCH_SIZE]]
            y = torch.tensor([LABELS.index(r["label"]) for r in rows], device=DEVICE)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(**encode(rows)).logits
            loss = loss_fn(logits.float(), y) / GRAD_ACCUM
            if not torch.isfinite(loss):
                raise FloatingPointError("loss is NaN")
            scaler.scale(loss).backward()
            losses.append(float(loss) * GRAD_ACCUM)
            if (i + 1) % GRAD_ACCUM == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()
        probs = dev_probs(model, dev)
        pred = [LABELS[int(np.argmax(p))] for p in probs]
        s = scores([r["label"] for r in dev], pred, LABELS)
        per_epoch.append({"epoch": epoch, "train_loss": float(np.mean(losses)), "dev_macro_f1": s["macro_f1"],
                          "dev_accuracy": s["accuracy"]})
        print(f"seed {seed} epoch {epoch}: loss {per_epoch[-1]['train_loss']:.3f} dev macro-F1 {s['macro_f1']:.3f}")
        if s["macro_f1"] > best[0]:
            best = (s["macro_f1"], epoch)
            model.save_pretrained(str(out_dir / "final")); tokenizer.save_pretrained(str(out_dir / "final"))
            best_scores, best_preds = s, [{"risk_id": r["risk_id"], "label": r["label"], "pred": p, "probs": q}
                                         for r, p, q in zip(dev, pred, probs)]
    metrics = {"seed": seed, "model": MODEL, "best_epoch": best[1], "epochs": EPOCHS, "smoke": SMOKE,
               "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE * GRAD_ACCUM, "max_len": MAX_LEN,
               "n_train": len(train), "n_dev": len(dev), "dev": best_scores, "per_epoch": per_epoch,
               "train_runtime_s": round(time.time() - t0),
               "gpu": torch.cuda.get_device_name(0) if use_amp else None}
    (Path(OUT_ROOT) / f"metrics-{seed}.json").write_text(json.dumps(metrics, indent=1), encoding="utf-8")
    (Path(OUT_ROOT) / f"dev-predictions-{seed}.json").write_text(json.dumps(best_preds), encoding="utf-8")
    del model; torch.cuda.empty_cache()
    return metrics

Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)
for seed in SEEDS:
    m = earlier_metrics(seed)
    if m and not m.get("smoke"):
        print(f"SEED {seed} already done (dev macro-F1 {m['dev']['macro_f1']:.3f}); skipped")
        continue
    m = run_seed(seed)
    print(f"SEED {seed} OK: best epoch {m['best_epoch']}, dev macro-F1 {m['dev']['macro_f1']:.3f}")
